In [ ]:
{
 "nbformat": 4,
 "nbformat_minor": 5,
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# FootballTacticsMARL – LLM Training\n",
    "Fine‑tune an LLM to become a football coach using supervised fine‑tuning on high‑reward actions."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!pip install -q unsloth trl transformers datasets accelerate torch numpy matplotlib openenv"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os, json, numpy as np, torch\n",
    "from unsloth import FastLanguageModel\n",
    "from transformers import TrainingArguments, Trainer\n",
    "from datasets import Dataset\n",
    "import sys\n",
    "# Mount Google Drive and add your environment path\n",
    "from google.colab import drive\n",
    "drive.mount('/content/drive')\n",
    "sys.path.insert(0, '/content/drive/MyDrive/playlogic')  # adjust to your uploaded folder\n",
    "from server.environment import FootballTacticsMARL\n",
    "from models import Action"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load model\n",
    "model, tokenizer = FastLanguageModel.from_pretrained(\n",
    "    model_name = \"unsloth/Llama-3.2-1B-Instruct\",\n",
    "    max_seq_length = 2048,\n",
    "    load_in_4bit = True,\n",
    ")\n",
    "model = FastLanguageModel.get_peft_model(model, r=16,\n",
    "    target_modules=[\"q_proj\",\"k_proj\",\"v_proj\",\"o_proj\"],\n",
    "    lora_alpha=16,\n",
    ")\n",
    "tokenizer.pad_token = tokenizer.eos_token"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Prompt template\n",
    "def build_prompt(obs_text):\n",
    "    return f\"\"\"You are controlling a football team. Given the current game state, decide actions for all 11 players. Output ONLY a valid JSON object mapping player ID to action as specified.\n",
    "Current state:\n",
    "{obs_text}\n",
    "Actions:\n",
    "\"\"\"\n",
    "\n",
    "def sample_action(obs_text):\n",
    "    prompt = build_prompt(obs_text)\n",
    "    inputs = tokenizer(prompt, return_tensors=\"pt\").to(\"cuda\")\n",
    "    with torch.no_grad():\n",
    "        outputs = model.generate(\n",
    "            **inputs,\n",
    "            max_new_tokens=256,\n",
    "            temperature=0.7,\n",
    "            do_sample=True,\n",
    "            pad_token_id=tokenizer.eos_token_id,\n",
    "        )\n",
    "    completion = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()\n",
    "    try:\n",
    "        start = completion.find('{')\n",
    "        end = completion.rfind('}')\n",
    "        if start != -1 and end != -1:\n",
    "            action_dict = json.loads(completion[start:end+1])\n",
    "            action_dict = {int(k): v for k, v in action_dict.items()}\n",
    "            return action_dict, completion[start:end+1]\n",
    "    except:\n",
    "        pass\n",
    "    # Fallback\n",
    "    return {i: {\"hold\": None} for i in range(11)}, json.dumps({i: {\"hold\": None} for i in range(11)})\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Training loop\n",
    "env = FootballTacticsMARL()\n",
    "all_rewards = []\n",
    "ITERATIONS = 5\n",
    "EPISODES_PER_ITER = 3\n",
    "\n",
    "for iteration in range(ITERATIONS):\n",
    "    print(f\"Iteration {iteration+1}\")\n",
    "    trajectories = []\n",
    "    for ep in range(EPISODES_PER_ITER):\n",
    "        obs = env.reset()\n",
    "        done = False\n",
    "        while not done:\n",
    "            pre_obs = obs.text\n",
    "            act_dict, act_str = sample_action(pre_obs)\n",
    "            obs, reward, done, info = env.step(Action(text=act_str))\n",
    "            trajectories.append({\n",
    "                \"prompt\": build_prompt(pre_obs),\n",
    "                \"completion\": act_str,\n",
    "                \"reward\": reward\n",
    "            })\n",
    "    avg_reward = sum(t['reward'] for t in trajectories) / len(trajectories)\n",
    "    all_rewards.append(avg_reward)\n",
    "    print(f\"Avg reward: {avg_reward:.3f}\")\n",
    "\n",
    "    # Keep only positive examples\n",
    "    positive = [t for t in trajectories if t['reward'] > 0]\n",
    "    if len(positive) == 0:\n",
    "        positive = trajectories\n",
    "    dataset = Dataset.from_list(positive)\n",
    "    def tokenize_fn(examples):\n",
    "        full = [p + c + tokenizer.eos_token for p, c in zip(examples[\"prompt\"], examples[\"completion\"])]\n",
    "        tok = tokenizer(full, truncation=True, max_length=2048, padding=\"max_length\")\n",
    "        tok[\"labels\"] = tok[\"input_ids\"].copy()\n",
    "        return tok\n",
    "    tokenized_dataset = dataset.map(tokenize_fn, batched=True)\n",
    "\n",
    "    args = TrainingArguments(\n",
    "        output_dir=f\"./tmp_{iteration}\",\n",
    "        per_device_train_batch_size=2,\n",
    "        num_train_epochs=2,\n",
    "        learning_rate=2e-4,\n",
    "        report_to=\"none\"\n",
    "    )\n",
    "    trainer = Trainer(model=model, args=args, train_dataset=tokenized_dataset, tokenizer=tokenizer)\n",
    "    trainer.train()\n",
    "\n",
    "# Save plot\n",
    "import matplotlib.pyplot as plt\n",
    "plt.plot(all_rewards, marker='o')\n",
    "plt.title(\"Average Reward per Step over Training\")\n",
    "plt.xlabel(\"Iteration\")\n",
    "plt.ylabel(\"Avg Reward\")\n",
    "plt.grid(True)\n",
    "plt.savefig(\"reward_curve.png\")\n",
    "plt.show()\n",
    "\n",
    "# Download the plot (it will be saved in the Colab runtime, then you can download)\n",
    "# Or save back to Drive\n",
    "# from google.colab import files\n",
    "# files.download(\"reward_curve.png\")"
   ]
  }
 ]
}